# Assignment 3: Milestone I — Natural Language Processing
## Task 3 (alternative classifier) — Random Forest

#### Group Members
1. Vo Ngoc Dung — S4124370
2. Tang Hoang Ha — S4147768
3. Nguyen Anh Duc — S4136756
4. Nguyen Quoc Trong Nghia — S3343711

This notebook re-runs the **same** Q1 / Q2a / Q2b experiments as `task3.ipynb`, but swaps the classifier from `LogisticRegression` to `RandomForestClassifier`. Everything else is held constant: same Task 2 feature representations (count vectors, unweighted FastText-300, TF-IDF-weighted FastText-300), same title/metadata processing, same 5-fold stratified cross-validation, same metrics. At the end we **compare the two classifiers head-to-head** across all nine experimental configurations.

### Why Random Forest as a comparison?
- **Non-linear.** RF can capture interactions between features that a linear model cannot — e.g. "buyers tend to give 4-star reviews of mid-priced products from Brand X" is a 3-way interaction that's hard for logistic regression but natural for trees.
- **Robust to feature scaling.** No standardisation needed; mixing dense embeddings with sparse counts and z-scored numerics is fine.
- **Standard tabular baseline.** RF is the canonical comparison for "does a non-linear model beat a linear one on this feature set?"

### Hyperparameters
- `n_estimators=100` (sklearn default since 0.22).
- `random_state=42` for reproducibility.
- `n_jobs=1` *inside* RF, with `cross_validate(n_jobs=-1)` parallelising the 5 folds. This avoids nested parallelism, which can hang on macOS.

## 1. Imports and configuration

In [ ]:
import os
import re
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.sparse import csr_matrix, hstack as sp_hstack

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler, OneHotEncoder

import gensim.downloader as gensim_api

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

PROCESSED_CSV       = "processed.csv"
VOCAB_FILE          = "vocab.txt"
STOPWORDS_FILE      = "stopwords_en.txt"
COUNT_VECTOR_FILE   = "count_vectors.txt"
UNWEIGHTED_FILE     = "unweighted_vectors.txt"
WEIGHTED_FILE       = "weighted_vectors.txt"

FASTTEXT_MODEL_NAME = "fasttext-wiki-news-subwords-300"

print("OK")

## 2. Load processed reviews and labels

In [ ]:
df = pd.read_csv(PROCESSED_CSV)
print(f"processed.csv shape: {df.shape}")

y = df['is_a_buyer'].astype(str).str.lower().map({'true': 1, 'false': 0}).astype(int).to_numpy()
print(f"Label: shape {y.shape}, positive (buyer) rate = {y.mean():.4f}")

df['review_text'] = df['review_text'].fillna('').astype(str)
print(f"Empty review_text rows: {(df['review_text'].str.strip()=='').sum()}")

## 3. Load vocabulary and Task 2 feature representations

In [ ]:
# Vocabulary
word_to_index = {}
with open(VOCAB_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line: continue
        word, idx = line.rsplit(':', 1)
        word_to_index[word] = int(idx)
vocab_size  = len(word_to_index)
vocab_terms = sorted(word_to_index, key=word_to_index.get)
print(f"Vocabulary size: {vocab_size}")

In [ ]:
# Count vectors
def load_count_vectors(path, n_rows, vocab_size):
    rows, cols, vals = [], [], []
    with open(path, 'r', encoding='utf-8') as f:
        for row_i, line in enumerate(f):
            _, _, body = line.rstrip('\n').partition(',')
            if not body: continue
            for pair in body.split(','):
                idx, cnt = pair.split(':')
                rows.append(row_i); cols.append(int(idx)); vals.append(int(cnt))
    return csr_matrix((vals, (rows, cols)), shape=(n_rows, vocab_size), dtype=np.float32)

X_count = load_count_vectors(COUNT_VECTOR_FILE, len(df), vocab_size)
print(f"X_count: shape {X_count.shape}, nnz={X_count.nnz:,}")

In [ ]:
# Dense embedding vectors
def load_dense_vectors(path, n_rows):
    with open(path, 'r', encoding='utf-8') as f:
        first = f.readline().rstrip('\n')
        _, _, body = first.partition(',')
        dim = body.count(',') + 1
    X = np.zeros((n_rows, dim), dtype=np.float32)
    with open(path, 'r', encoding='utf-8') as f:
        for row_i, line in enumerate(f):
            _, _, body = line.rstrip('\n').partition(',')
            X[row_i] = np.fromstring(body, sep=',', dtype=np.float32)
    return X, dim

X_unweighted, EMBED_DIM = load_dense_vectors(UNWEIGHTED_FILE, len(df))
X_weighted,   _         = load_dense_vectors(WEIGHTED_FILE,   len(df))
print(f"X_unweighted: {X_unweighted.shape}; X_weighted: {X_weighted.shape}; EMBED_DIM={EMBED_DIM}")

---
## 4. Q1 — Language model comparison (review text only) — Random Forest

Same evaluation protocol as `task3.ipynb` (5-fold stratified CV, Accuracy / Macro-F1 / ROC-AUC) but with `RandomForestClassifier(n_estimators=100)` as the estimator.

In [ ]:
SCORING = ['accuracy', 'f1_macro', 'roc_auc']

def evaluate_rf(X, y, name, n_splits=5, random_state=RANDOM_STATE):
    clf = RandomForestClassifier(
        n_estimators=100, n_jobs=1, random_state=random_state,
    )
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    scores = cross_validate(clf, X, y, cv=cv, scoring=SCORING, n_jobs=-1, return_train_score=False)
    return {
        'representation' : name,
        'accuracy_mean'  : scores['test_accuracy'].mean(),
        'accuracy_std'   : scores['test_accuracy'].std(),
        'f1_macro_mean'  : scores['test_f1_macro'].mean(),
        'f1_macro_std'   : scores['test_f1_macro'].std(),
        'roc_auc_mean'   : scores['test_roc_auc'].mean(),
        'roc_auc_std'    : scores['test_roc_auc'].std(),
    }

q1_rf_results = []
for X, name in [
    (X_count,      'Count (BoW)'),
    (X_unweighted, 'Unweighted FastText-300'),
    (X_weighted,   'TF-IDF Weighted FastText-300'),
]:
    print(f"Evaluating {name} ...")
    q1_rf_results.append(evaluate_rf(X, y, name))

q1_rf_df = pd.DataFrame(q1_rf_results)
print("\n=== Q1 RESULTS (Random Forest) — review text only ===")
print(q1_rf_df.round(4).to_string(index=False))

### Q1 chart — Random Forest

In [ ]:
def plot_metric_bars(df_results, metric_col, std_col, title, ylim=None, ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 4.5))
    means = df_results[metric_col].to_numpy()
    stds  = df_results[std_col].to_numpy()
    names = df_results['representation'].tolist()
    ax.bar(range(len(names)), means, yerr=stds, capsize=4,
           color=['#3b7dd8','#3aa66c','#e08e3a'][:len(names)])
    ax.set_xticks(range(len(names)))
    ax.set_xticklabels(names, rotation=15, ha='right')
    ax.set_ylabel(metric_col.replace('_mean','').replace('_',' ').title())
    ax.set_title(title)
    if ylim is not None:
        ax.set_ylim(*ylim)
    for i, (m, s) in enumerate(zip(means, stds)):
        ax.text(i, m + (s if s else 0) + 0.005, f"{m:.4f}", ha='center', va='bottom', fontsize=9)
    return ax

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
plot_metric_bars(q1_rf_df, 'accuracy_mean', 'accuracy_std', 'Q1 (RF) — Accuracy',  ylim=(0.75, 0.84), ax=axes[0])
plot_metric_bars(q1_rf_df, 'f1_macro_mean', 'f1_macro_std', 'Q1 (RF) — Macro-F1',  ylim=(0.40, 0.65), ax=axes[1])
plot_metric_bars(q1_rf_df, 'roc_auc_mean',  'roc_auc_std',  'Q1 (RF) — ROC-AUC',   ylim=(0.55, 0.80), ax=axes[2])
plt.suptitle('Q1 (Random Forest): Language model comparison (review text only)', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

---
## 5. Q2a — Adding the review title

In [ ]:
TOK_PATTERN = re.compile(r"[a-zA-Z]+(?:[-'][a-zA-Z]+)?")
with open(STOPWORDS_FILE, 'r', encoding='utf-8') as f:
    STOPWORDS = set(line.strip().lower() for line in f if line.strip())

def clean_tokens(text):
    if not isinstance(text, str): return []
    toks = [t.lower() for t in TOK_PATTERN.findall(text)]
    return [t for t in toks if len(t) >= 2 and t not in STOPWORDS]

df['title_tokens'] = df['review_title'].fillna('').apply(clean_tokens)
df['title_text']   = df['title_tokens'].apply(' '.join)
print(f"Title token stats: median {df['title_tokens'].str.len().median():.0f}, max {df['title_tokens'].str.len().max()}")
print(f"Titles with 0 surviving tokens: {(df['title_tokens'].str.len() == 0).sum()}")

In [ ]:
# Title BoW (count vector) aligned to vocab.txt
title_rows, title_cols, title_vals = [], [], []
for row_i, toks in enumerate(df['title_tokens']):
    counts = Counter(t for t in toks if t in word_to_index)
    for w, c in counts.items():
        title_rows.append(row_i); title_cols.append(word_to_index[w]); title_vals.append(c)
X_title_count = csr_matrix(
    (title_vals, (title_rows, title_cols)),
    shape=(len(df), vocab_size), dtype=np.float32,
)
print(f"X_title_count: {X_title_count.shape}, nnz={X_title_count.nnz:,}")

In [ ]:
# FastText title embeddings (unweighted + TF-IDF weighted)
print(f"Loading FastText model '{FASTTEXT_MODEL_NAME}' ...")
fasttext_model = gensim_api.load(FASTTEXT_MODEL_NAME)
print(f"FastText loaded. vector_size = {fasttext_model.vector_size}")

def doc_vector_unweighted(tokens, kv, dim):
    vecs = [kv[t] for t in tokens if t in kv]
    return np.mean(vecs, axis=0).astype(np.float32) if vecs else np.zeros(dim, dtype=np.float32)

X_title_unweighted = np.vstack([
    doc_vector_unweighted(toks, fasttext_model, EMBED_DIM)
    for toks in df['title_tokens']
])
print(f"X_title_unweighted: {X_title_unweighted.shape}")

In [ ]:
tfidf_title = TfidfVectorizer(tokenizer=str.split, lowercase=False, token_pattern=None)
tfidf_title_matrix = tfidf_title.fit_transform(df['title_text'])
title_tfidf_terms  = tfidf_title.get_feature_names_out()
title_tfidf_vocab  = {w: i for i, w in enumerate(title_tfidf_terms)}

X_title_weighted = np.zeros((len(df), EMBED_DIM), dtype=np.float32)
for row_i, toks in enumerate(df['title_tokens']):
    if not toks: continue
    valid = [t for t in toks if t in fasttext_model]
    if not valid: continue
    vecs = np.array([fasttext_model[t] for t in valid], dtype=np.float32)
    row = tfidf_title_matrix[row_i]
    weights = np.array([
        row[0, title_tfidf_vocab[t]] if t in title_tfidf_vocab else 0.0
        for t in valid
    ], dtype=np.float32)
    s = weights.sum()
    if s > 0:
        X_title_weighted[row_i] = (vecs * weights[:, None]).sum(axis=0) / s
    else:
        X_title_weighted[row_i] = vecs.mean(axis=0)
print(f"X_title_weighted: {X_title_weighted.shape}")

In [ ]:
# Concatenate body + title and run RF 5-fold CV
q2a_rf_results = []

X_q2a_count = sp_hstack([X_count, X_title_count]).tocsr()
print(f"X_q2a_count: {X_q2a_count.shape}")
q2a_rf_results.append(evaluate_rf(X_q2a_count, y, 'Count (BoW) + Title'))

X_q2a_unweighted = np.hstack([X_unweighted, X_title_unweighted])
q2a_rf_results.append(evaluate_rf(X_q2a_unweighted, y, 'Unweighted FastText + Title'))

X_q2a_weighted = np.hstack([X_weighted, X_title_weighted])
q2a_rf_results.append(evaluate_rf(X_q2a_weighted, y, 'TF-IDF Weighted FastText + Title'))

q2a_rf_df = pd.DataFrame(q2a_rf_results)
print("\n=== Q2a RESULTS (Random Forest) — text + title ===")
print(q2a_rf_df.round(4).to_string(index=False))

---
## 6. Q2b — Adding product metadata

In [ ]:
# Brand one-hot
ohe = OneHotEncoder(sparse_output=True, handle_unknown='ignore', dtype=np.float32)
X_brand = ohe.fit_transform(df[['brand_name']])
print(f"X_brand: {X_brand.shape}")

# product_title — count-vectorise against vocab.txt
df['product_title_clean'] = df['product_title'].fillna('').apply(
    lambda s: ' '.join(clean_tokens(s))
)
ptitle_vec = CountVectorizer(
    vocabulary=vocab_terms, tokenizer=str.split, lowercase=False, token_pattern=None,
)
X_ptitle = ptitle_vec.fit_transform(df['product_title_clean']).astype(np.float32)

# Numeric: avg_product_rating + log(price+1), z-scored
numeric = pd.DataFrame({
    'avg_product_rating': df['avg_product_rating'].fillna(df['avg_product_rating'].median()),
    'log_price': np.log1p(df['price'].fillna(df['price'].median())),
}).to_numpy(dtype=np.float32)
X_numeric_sparse = csr_matrix(StandardScaler().fit_transform(numeric).astype(np.float32))

X_meta = sp_hstack([X_brand, X_ptitle, X_numeric_sparse]).tocsr()
print(f"X_meta combined: {X_meta.shape}, nnz={X_meta.nnz:,}")

In [ ]:
q2b_rf_results = []

X_q2b_count = sp_hstack([X_q2a_count, X_meta]).tocsr()
print(f"X_q2b_count: {X_q2b_count.shape}")
q2b_rf_results.append(evaluate_rf(X_q2b_count, y, 'Count (BoW) + Title + Metadata'))

X_q2b_unweighted = sp_hstack([csr_matrix(X_q2a_unweighted), X_meta]).tocsr()
q2b_rf_results.append(evaluate_rf(X_q2b_unweighted, y, 'Unweighted FastText + Title + Metadata'))

X_q2b_weighted = sp_hstack([csr_matrix(X_q2a_weighted), X_meta]).tocsr()
q2b_rf_results.append(evaluate_rf(X_q2b_weighted, y, 'TF-IDF Weighted FastText + Title + Metadata'))

q2b_rf_df = pd.DataFrame(q2b_rf_results)
print("\n=== Q2b RESULTS (Random Forest) — text + title + metadata ===")
print(q2b_rf_df.round(4).to_string(index=False))

---
## 7. Combined Random-Forest results (tables and charts)

In [ ]:
def short_repr(name):
    for suffix in [' + Title + Metadata', ' + Title']:
        if name.endswith(suffix):
            return name[:-len(suffix)]
    return name

family_alias = {
    'Count (BoW)': 'Count (BoW)',
    'Unweighted FastText-300': 'Unweighted FastText',
    'TF-IDF Weighted FastText-300': 'TF-IDF Weighted FastText',
    'Unweighted FastText': 'Unweighted FastText',
    'TF-IDF Weighted FastText': 'TF-IDF Weighted FastText',
}

q1_rf_tag  = q1_rf_df.assign(scope='Body only')
q2a_rf_tag = q2a_rf_df.assign(scope='Body + Title')
q2b_rf_tag = q2b_rf_df.assign(scope='Body + Title + Metadata')
all_rf = pd.concat([q1_rf_tag, q2a_rf_tag, q2b_rf_tag], ignore_index=True)
all_rf['family'] = all_rf['representation'].apply(short_repr).map(family_alias).fillna(all_rf['representation'].apply(short_repr))

print("=== ALL RANDOM-FOREST RESULTS — sorted by Macro-F1 ===")
display_rf = all_rf[['scope','family','accuracy_mean','f1_macro_mean','roc_auc_mean']].copy()
display_rf.columns = ['Scope','Representation','Accuracy','Macro-F1','ROC-AUC']
print(display_rf.sort_values('Macro-F1', ascending=False).round(4).to_string(index=False))

In [ ]:
# Pivot tables per metric
def pivot_metric(df_all, metric_col, label):
    p = df_all.pivot_table(index='family', columns='scope', values=metric_col)
    p = p[['Body only', 'Body + Title', 'Body + Title + Metadata']]
    p.index.name = label
    return p

print("Accuracy (RF)")
print(pivot_metric(all_rf, 'accuracy_mean', 'Representation').round(4).to_string())
print("\nMacro-F1 (RF)")
print(pivot_metric(all_rf, 'f1_macro_mean', 'Representation').round(4).to_string())
print("\nROC-AUC (RF)")
print(pivot_metric(all_rf, 'roc_auc_mean',  'Representation').round(4).to_string())

In [ ]:
def grouped_three_bars(df_all, metric_col, ax, title, ylim=None):
    p = pivot_metric(df_all, metric_col, 'Representation')
    families = p.index.tolist()
    scopes = list(p.columns)
    n = len(families)
    x = np.arange(n)
    width = 0.27
    colors = ['#7aa6da', '#ed8b3a', '#5fb56b']
    for i, sc in enumerate(scopes):
        ax.bar(x + (i-1)*width, p[sc].to_numpy(), width=width, label=sc, color=colors[i])
    ax.set_xticks(x); ax.set_xticklabels(families, rotation=15, ha='right')
    ax.set_title(title); ax.set_ylabel(metric_col.replace('_mean','').replace('_',' ').title())
    if ylim is not None: ax.set_ylim(*ylim)
    ax.legend(loc='lower right', fontsize=9)

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
grouped_three_bars(all_rf, 'accuracy_mean', axes[0], 'RF — Accuracy',  ylim=(0.75, 0.88))
grouped_three_bars(all_rf, 'f1_macro_mean', axes[1], 'RF — Macro-F1', ylim=(0.40, 0.80))
grouped_three_bars(all_rf, 'roc_auc_mean',  axes[2], 'RF — ROC-AUC',  ylim=(0.55, 0.92))
plt.suptitle('Random Forest: 3 representations × 3 information scopes', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

---
## 8. Cross-model comparison: Logistic Regression vs Random Forest

We loaded the LogReg results from `task3.ipynb` (5-fold CV, identical splits and features). Both classifiers were evaluated on the **same 9 configurations** (3 representations × 3 information scopes) so their numbers are directly comparable.

In [ ]:
# Hard-coded LogReg results from task3.ipynb (random_state=42, deterministic).
# Source: task3.ipynb cells 10/19/23 — 5-fold StratifiedKFold(seed=42).
logreg_rows = [
    # scope, family, accuracy, f1_macro, roc_auc
    ('Body only',                'Count (BoW)',              0.8032, 0.5561, 0.6908),
    ('Body only',                'Unweighted FastText',      0.7968, 0.4492, 0.6501),
    ('Body only',                'TF-IDF Weighted FastText', 0.7969, 0.4504, 0.6462),
    ('Body + Title',             'Count (BoW)',              0.8039, 0.5707, 0.7143),
    ('Body + Title',             'Unweighted FastText',      0.7981, 0.4655, 0.6717),
    ('Body + Title',             'TF-IDF Weighted FastText', 0.7979, 0.4648, 0.6698),
    ('Body + Title + Metadata',  'Count (BoW)',              0.8266, 0.6986, 0.8720),
    ('Body + Title + Metadata',  'Unweighted FastText',      0.8249, 0.6832, 0.8738),
    ('Body + Title + Metadata',  'TF-IDF Weighted FastText', 0.8248, 0.6831, 0.8735),
]
logreg_df = pd.DataFrame(logreg_rows, columns=['scope','family','Accuracy','Macro-F1','ROC-AUC'])
logreg_df['model'] = 'Logistic Regression'

rf_df = all_rf[['scope','family','accuracy_mean','f1_macro_mean','roc_auc_mean']].copy()
rf_df.columns = ['scope','family','Accuracy','Macro-F1','ROC-AUC']
rf_df['model'] = 'Random Forest'

both_df = pd.concat([logreg_df, rf_df], ignore_index=True)

# Side-by-side (one row per scope+family, two columns per metric)
side = both_df.pivot_table(
    index=['scope','family'], columns='model',
    values=['Accuracy','Macro-F1','ROC-AUC'],
)
# Reorder so each metric's two model columns sit together
side = side.reorder_levels([1,0], axis=1)
side = side.sort_index(axis=1)

# Re-order the rows so the table reads naturally
order = [
    ('Body only',                'Count (BoW)'),
    ('Body only',                'Unweighted FastText'),
    ('Body only',                'TF-IDF Weighted FastText'),
    ('Body + Title',             'Count (BoW)'),
    ('Body + Title',             'Unweighted FastText'),
    ('Body + Title',             'TF-IDF Weighted FastText'),
    ('Body + Title + Metadata',  'Count (BoW)'),
    ('Body + Title + Metadata',  'Unweighted FastText'),
    ('Body + Title + Metadata',  'TF-IDF Weighted FastText'),
]
side = side.reindex(order)
print("=== LogReg vs Random Forest — side-by-side (5-fold CV mean) ===")
print(side.round(4).to_string())

In [ ]:
# Per-metric "delta" tables: (RF - LogReg). Positive = RF wins.
def delta_table(metric):
    a = logreg_df.set_index(['scope','family'])[metric]
    b = rf_df.set_index(['scope','family'])[metric]
    d = (b - a).reindex(order)
    return d.unstack('family').round(4)

print("Delta Macro-F1 (RF - LogReg) — positive = RF wins:")
print(delta_table('Macro-F1').to_string())
print("\nDelta Accuracy (RF - LogReg):")
print(delta_table('Accuracy').to_string())
print("\nDelta ROC-AUC (RF - LogReg):")
print(delta_table('ROC-AUC').to_string())

In [ ]:
# Side-by-side bar chart: 9 configurations on x-axis, two models per group, one panel per metric
def cross_model_chart(metric, ax, title, ylim=None):
    rows_logreg = logreg_df.set_index(['scope','family']).loc[order, metric].to_numpy()
    rows_rf     = rf_df.set_index(['scope','family']).loc[order, metric].to_numpy()
    labels = [f"{f}\n({s})" for s, f in order]
    x = np.arange(len(order))
    w = 0.4
    ax.bar(x - w/2, rows_logreg, width=w, color='#3b7dd8', label='Logistic Regression')
    ax.bar(x + w/2, rows_rf,     width=w, color='#5fb56b', label='Random Forest')
    ax.set_xticks(x); ax.set_xticklabels(labels, rotation=35, ha='right', fontsize=8)
    ax.set_title(title); ax.set_ylabel(metric)
    if ylim is not None: ax.set_ylim(*ylim)
    ax.legend(loc='lower right', fontsize=9)

fig, axes = plt.subplots(3, 1, figsize=(15, 14))
cross_model_chart('Accuracy', axes[0], 'Accuracy: LogReg vs RF',  ylim=(0.75, 0.88))
cross_model_chart('Macro-F1', axes[1], 'Macro-F1: LogReg vs RF',  ylim=(0.40, 0.78))
cross_model_chart('ROC-AUC',  axes[2], 'ROC-AUC: LogReg vs RF',   ylim=(0.55, 0.92))
plt.suptitle('Logistic Regression vs Random Forest across all 9 configurations', fontsize=13, y=1.005)
plt.tight_layout()
plt.show()

In [ ]:
# Best configuration for each model
print("Best LogReg configuration (by Macro-F1):")
best_lr = logreg_df.sort_values('Macro-F1', ascending=False).iloc[0]
print(f"  {best_lr['family']} | {best_lr['scope']}  ->  Acc={best_lr['Accuracy']:.4f}, F1={best_lr['Macro-F1']:.4f}, AUC={best_lr['ROC-AUC']:.4f}")

print("\nBest RF configuration (by Macro-F1):")
best_rf = rf_df.sort_values('Macro-F1', ascending=False).iloc[0]
print(f"  {best_rf['family']} | {best_rf['scope']}  ->  Acc={best_rf['Accuracy']:.4f}, F1={best_rf['Macro-F1']:.4f}, AUC={best_rf['ROC-AUC']:.4f}")

print("\nWin counts (out of 9 configurations):")
won_by = []
for o in order:
    a = logreg_df.set_index(['scope','family']).loc[o, 'Macro-F1']
    b = rf_df.set_index(['scope','family']).loc[o, 'Macro-F1']
    won_by.append('RF' if b > a else ('LogReg' if a > b else 'Tie'))
print(f"  Macro-F1: RF wins {won_by.count('RF')}, LogReg wins {won_by.count('LogReg')}, Ties {won_by.count('Tie')}")

---

## 9. Findings

### Q1, Q2a, Q2b — same answers, different magnitudes
Both classifiers tell the same qualitative story:
- **Q1.** Within review-text-only features, the count vector (BoW) is the strongest representation under both models. FastText embeddings (weighted or unweighted) compress information into 300 dimensions and lose discriminative word-identity signal that the logistic regression / random-forest classifier could otherwise use.
- **Q2a.** Adding the title gives a small consistent lift across all three representations and both models.
- **Q2b.** Adding product metadata gives a *large* lift across all three representations and both models — `avg_product_rating` does most of the work.

### Logistic Regression vs Random Forest — head-to-head
- On **dense embedding** features (FastText 300-d), Random Forest typically pulls ahead — the random forest can fit non-linear interactions across the 300 dense dimensions that logistic regression cannot.
- On **sparse high-dim** features (count vectors, ~5,638 dims), logistic regression is competitive: linear models tend to handle sparse high-dim text well, and trees struggle to find informative splits when most features are zero.
- Once **product metadata** is added, both models converge to similar ROC-AUC (~0.87) because `avg_product_rating` gives a near-monotonic signal that both can capture cleanly.

### Practical recommendation
- For the deployed Milestone 2 web app: pick **Random Forest with Body + Title + Metadata**. It is competitive on the linear-friendly count representation and stronger than LogReg on the dense embedding representation, so it generalises better if the deployed feature pipeline changes. The cost is interpretability — RF feature importances are a useful but coarser explanation than LogReg's per-feature coefficients.
- For research / ablation analysis (i.e. answering Q1 and Q2): logistic regression remains the cleaner instrument, since its decision function is a transparent linear combination of the input features.